# Chapter 09 — Shells, Processes, and Environment Variables

**Companion to *Pi Agents*** · [`Pi Agents` chapter 09](https://programmer.ie/books/pi/09-chapter/)

| | |
|---|---|
| Chapter | `09-chapter.md` · weight 9 · `/books/pi/09-chapter/` |
| Notebook | `notebooks/pi/09-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

When the model runs `bash` and when **you** type `!`, are those the same
shell with the same variables? **Compare the two environments directly.**

The chapter's title case is an alias: something that works when you type it, failing
when the model runs it, with no error message that explains why.


## What this notebook establishes

- The model's `bash` tool receives the **session variables** — `PI_SESSION_ID`, `PI_PROVIDER`, `PI_MODEL`, `PI_REASONING_LEVEL`, `PI_SESSION_FILE`. The operator's `!` command does **not**: `PI_SESSION_ID` is empty there.
- `PI_SESSION_FILE` is populated for a persisted session and empty for an ephemeral one, which is the documented way for a script to tell.
- `shellCommandPrefix` runs before **both** entry points.
- A non-zero exit is reported as a result the model can read: the output is in the result text and `isError` is true.

## What this notebook does **not** establish

- **The shell is whatever Pi resolved on this machine.** The run below records the resolved `bash` path. The chapter states the same caveat: these ran under Git Bash on Windows, and on another platform the shell — and therefore the alias result — may differ.
- **No container, no isolation.** `metadata/09-chapter.yaml` records containerization as out of scope; it is chapter 42's subject.
- **Nothing about `powershell` / `pwsh`.** Those are available only to a native Windows Pi process, while your own `!` commands keep using Bash.
- **`AI_AGENT=pi` and `PI_CODING_AGENT=true`** are set by the CLI and RPC entry points only, not automatically through the SDK, and are not read here.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the same command string through the model's `bash` tool and through `session.executeBash`, in one real session.
* **Evidence scope:** `in_process_runtime`. The shell is the reader's; the session variables and the prefix handling are Pi's.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(9))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/09-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Experiment 1: two callers, two environments

One session. One command string that prints the session variables. Run once as the
model's tool call and once as the operator's `!` command.

The interesting cells are the two that are *empty*: they are the boundary.

In [3]:
SHELL = pinb.driver_source("ch09-shell.ts")
shell = pinb.driver(SHELL, label="ch09-shell", timeout=300)
vars_row = shell[0]
plain = next(r for r in shell if r["label"] == "a plain echo")
failing = next(r for r in shell if r["label"] == "a command that exits 3")
prefixed = next(r for r in shell if r["label"] == "with shellCommandPrefix set")

print("The same command, two callers:")
print(pinb.md_table(
    ["variable", "the model's bash tool", "your ! command"],
    [["PI_SESSION_ID", vars_row["id"], vars_row["bangId"]],
     ["PI_SESSION_FILE", vars_row["file"], vars_row["bangFile"]]],
))
print()
print(pinb.md_table(
    ["case", "via the model", "via !", "identical?"],
    [[plain["label"], plain["viaModel"]["text"], plain["viaBang"]["output"], plain["sameOutput"]],
     [prefixed["label"], prefixed["viaModel"]["text"], prefixed["viaBang"]["output"], prefixed["sameOutput"]]],
))

The same command, two callers:
| variable | the model's bash tool | your ! command |
|---|---|---|
| PI_SESSION_ID | id=01a11843-9cdb-7314-821c-3a19d0e2bbab | (no id) |
| PI_SESSION_FILE | file=<tmp> | file= |

| case | via the model | via ! | identical? |
|---|---|---|---|
| a plain echo | hello-from-both | hello-from-both | True |
| with shellCommandPrefix set | yes | yes | True |


In [4]:
pinb.show_checks([
    pinb.check("the model's tool sees PI_SESSION_ID", vars_row["modelSawSessionId"], vars_row["id"]),
    pinb.check("your ! command does not", not vars_row["bangSawSessionId"], vars_row["bangId"]),
    pinb.check("PI_SESSION_FILE is populated for a persisted session, in the model's tool only",
               vars_row["modelSawSessionFile"] and not vars_row["bangSawSessionFile"], vars_row["file"]),
    pinb.check("an ordinary command produces identical output either way", plain["sameOutput"], plain["viaModel"]["text"]),
    pinb.check("a non-zero exit is a readable result, and isError is true",
               failing["isError"] is True and "about-to-fail" in failing["text"] and "3" in failing["text"],
               failing["text"].replace(chr(10), " | ")),
    pinb.check("shellCommandPrefix reaches both entry points",
               prefixed["sameOutput"] and prefixed["viaModel"]["text"] == "yes",
               "FROM_PREFIX was visible to the model and to !"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------------------------  ------------------------------------------------
PASS  the model's tool sees PI_SESSION_ID  id=01a11843-9cdb-7314-821c-3a19d0e2bbab
PASS  your ! command does not  (no id)
PASS  PI_SESSION_FILE is populated for a persisted session, in the model's tool…  file=<tmp>
PASS  an ordinary command produces identical output either way  hello-from-both
PASS  a non-zero exit is a readable result, and isError is true  about-to-fail |  |  | Command exited with code 3
PASS  shellCommandPrefix reaches both entry points  FROM_PREFIX was visible to the model and to !

6/6 assertions held.


The asymmetry is not a bug and it is not subtle once you see it: the session
variables are injected for **the model's** subprocess, not for yours.

## Contrasting case: the alias

The chapter's title case, and the most useful thing on this page. An alias defined
in an operator's interactive profile does not exist in Pi's fresh non-interactive
shell — and once `shellCommandPrefix` enables alias expansion and defines it, it
does.

The driver records which shell was used, so the result can be read with its
platform caveat attached.

In [5]:
ALIAS = pinb.driver_source("ch09-alias.ts")
alias = pinb.driver(ALIAS, label="ch09-alias", timeout=300)
print("shell Pi resolved:", alias["shell"]["bashPath"], "| usable:", alias["shell"]["bashWorks"])
print()
print(pinb.md_table(
    ["configuration", "the model runs `myalias`", "isError"],
    [["default", alias["withoutPrefix"]["text"].splitlines()[0], alias["withoutPrefix"]["isError"]],
     ["shellCommandPrefix enables + defines", alias["withPrefix"]["text"].splitlines()[0], alias["withPrefix"]["isError"]]],
))

if not alias["shell"]["bashWorks"]:
    pinb.unrun("the alias case", "no usable POSIX shell (Git Bash or bash on PATH)")
else:
    pinb.show_checks([
        pinb.check("without a prefix the alias does not exist",
                   alias["withoutPrefix"]["isError"] is True, alias["withoutPrefix"]["text"].splitlines()[0]),
        pinb.check("with the prefix it resolves",
                   alias["withPrefix"]["isError"] is False and "aliased-ok" in alias["withPrefix"]["text"],
                   alias["withPrefix"]["text"].splitlines()[0]),
    ], strict=False)
    assert alias["withPrefix"]["isError"] is False

shell Pi resolved: C:\Program Files\Git\bin\bash.exe | usable: True

| configuration | the model runs `myalias` | isError |
|---|---|---|
| default | /usr/bin/bash: line 1: myalias: command not found | True |
| shellCommandPrefix enables + defines | aliased-ok | False |
== Assertions ==
  assertion  observed
----  -----------------------------------------  -------------------------------------------------
PASS  without a prefix the alias does not exist  /usr/bin/bash: line 1: myalias: command not found
PASS  with the prefix it resolves  aliased-ok

2/2 assertions held.


## Your turn: predict, then run

**Predict first.** `shellCommandPrefix: "export FROM_PREFIX=yes"` was visible to
both callers above. What about `cd /somewhere` — would *your* next `!` command also
start in `/somewhere`, or is the prefix per-process?

**Then change one input.** `PREFIX` below is what Pi runs before each command. Try
one that **fails**: `exit 1`.

**Predict the boundary.** If the prefix exits non-zero, does the command still
run? What does the model see?

In [6]:
PREFIX = "exit 1"
COMMAND = "echo after-the-prefix"
print(f"predicted: with prefix {PREFIX!r}, {COMMAND!r} does not run and the model sees the failure")
print()
broken = pinb.driver(
    ALIAS,
    label="ch09-broken-prefix",
    timeout=300,
    env={"NB_COMMAND": COMMAND, "NB_PREFIX": PREFIX},
)
print("prefix:", repr(PREFIX), " command:", repr(COMMAND))
print("without a prefix: isError =", broken["withoutPrefix"]["isError"], "text =", repr(broken["withoutPrefix"]["text"][:120]))
print("with that prefix: isError =", broken["withPrefix"]["isError"], "text =", repr(broken["withPrefix"]["text"][:120]))

assert broken["withPrefix"]["isError"] is True, "a failing prefix fails the command"
assert broken["withoutPrefix"]["isError"] is False and "after-the-prefix" in broken["withoutPrefix"]["text"]
print()
print("held: the prefix is part of the command, so its failure is the command's failure")

predicted: with prefix 'exit 1', 'echo after-the-prefix' does not run and the model sees the failure



prefix: 'exit 1'  command: 'echo after-the-prefix'
without a prefix: isError = False text = 'after-the-prefix'
with that prefix: isError = True text = '(no output)\n\nCommand exited with code 1'

held: the prefix is part of the command, so its failure is the command's failure


## Interpretation

Three rules, all visible above:

1. **Session variables are for the model's subprocess.** If you want to know the
   session id from your own `!` command, read `session.sessionId` in an extension —
   or use a custom command, which is what chapters 12 and 20 are for.
2. **The shell is fresh and non-interactive.** Nothing from your interactive profile
   is there: no aliases, no `~/.bashrc`, no exported functions. `shellCommandPrefix`
   is the documented seam for putting something back.
3. **A non-zero exit is readable data, not an opaque failure.** The command's output
   is in the result and the model can act on it. Chapter 17 makes the same point
   about tools in general.

### What this chapter does not settle

The platform caveat is real and the chapter states it: these results are from Git
Bash on Windows, and the shell Pi resolves — and therefore the alias row — may
differ on your machine. The driver prints the resolved path so you can tell which
shell produced the result you are looking at.

## Sources, execution mode and limitations

**Execution mode:** **Run** — real subprocesses launched by Pi's own `bash` tool and by `session.executeBash`, plus the chapter's own test file.

**Evidence scope:** `in_process_runtime`. The shell is the reader's; the variables, the prefix and the result shape are Pi's.

### What was read or run

- **Ran** `drivers/ch09-shell.ts` and `drivers/ch09-alias.ts`, each creating its own sessions with their own temporary agent directories.
- **Ran** `ch09-shell/shell.test.ts`, which covers the thinking-level change and the ephemeral `PI_SESSION_FILE` rows that this driver does not repeat.
- **Read** `examples/evidence.json`, chapter 9 rows (7 claims).

### Limitations

- **Platform-dependent by nature.** `bashPath()` and `bashWorks()` are read from the book's own helper and printed above; the alias result depends on which shell that is.
- **`powershell` and `pwsh` were not run.** They are available only to a native Windows Pi process, and the operator's own `!` commands keep using Bash, so the two-caller comparison here is Bash-only.
- **The thinking-level row is not repeated here** — a session variable resolved at command start is asserted by the chapter's own test, which ran above.
- **No isolation.** Commands run with the notebook's own permissions in a temporary directory.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
